# Prophet baseline (with exogenous regressors)

Prophet decomposes the series into trend + daily/weekly/yearly seasonality + holidays + exogenous regressors. We add a curated set of regressors via `.add_regressor(...)` so this is more comparable to the supervised baselines than pure foundation-model notebooks.

In [ ]:
# Cell 1 - setup
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from models.utils import (
    chronological_split, select_enhanced_features,
    TARGET_REG, TARGET_CLASS,
    regression_report, classification_report,
    apply_feature_transforms,
)
FEATURES = PROJECT_ROOT / "data" / "features"


In [ ]:
# Cell 2 - load & split
mat = pd.read_parquet(FEATURES / "feature_matrix_engineered_v2.parquet")
mat = apply_feature_transforms(mat)
train, val, test = chronological_split(mat)
features = select_enhanced_features(mat)
print(f"{len(features)} features  train={len(train):,}  val={len(val):,}  test={len(test):,}")


In [ ]:
!pip install prophet --quiet

## Regressor selection

Curated set of 10 features chosen to mirror the dominant signals in the XGBoost baseline (short/medium/long-horizon price lags, weather, load, gas, day-ahead price, hour-of-day, peak-hour flag). Names verified against `select_enhanced_features(mat)`.

In [ ]:
REGRESSORS = ['hubavg_lag_1h', 'hubavg_lag_24h', 'hubavg_lag_168h', 'temp_max_across_zones', 'wind_mean_across_zones', 'load_actual_mw', 'henry_hub_price', 'da_HB_HUBAVG', 'hour', 'is_peak_hours']
missing = [c for c in REGRESSORS if c not in mat.columns]
assert not missing, f"Missing regressors: {missing}"
print(f"using {len(REGRESSORS)} regressors:")
for r in REGRESSORS:
    print(f"  - {r}")


In [ ]:
# Prophet wants ds, y plus regressor columns. Build a single frame and split chronologically.
needed_cols = [TARGET_REG] + REGRESSORS
df_prophet = mat[needed_cols].dropna().copy()
df_prophet = df_prophet.reset_index().rename(columns={df_prophet.index.name or "datetime": "ds",
                                                     TARGET_REG: "y"})
# Prophet requires tz-naive timestamps.
df_prophet["ds"] = pd.to_datetime(df_prophet["ds"]).dt.tz_localize(None)

# Reuse the chronological_split boundaries: train+val = fit, test = forecast.
_, _, test_split = chronological_split(mat)
test_start = pd.to_datetime(test_split.index.min()).tz_localize(None)

train_val_df = df_prophet[df_prophet["ds"] < test_start].copy()
test_df = df_prophet[df_prophet["ds"] >= test_start].copy()
print(f"train+val rows: {len(train_val_df):,}  test rows: {len(test_df):,}")
print(f"test horizon: {test_df['ds'].min()} -> {test_df['ds'].max()}")


In [ ]:
# Fit Prophet with daily/weekly/yearly seasonality and the curated regressors.
# NOTE: 60k+ hourly rows can take 10-30 minutes to fit on CPU. If runtime is prohibitive,
# uncomment the 3-year-window line below for a faster fit.
from prophet import Prophet

# train_val_df = train_val_df[train_val_df["ds"] >= "2020-01-01"]  # faster fit fallback

m = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=True,
)
for r in REGRESSORS:
    m.add_regressor(r)

m.fit(train_val_df)
print("Prophet fit complete.")


In [ ]:
# Forecast over the 2024 test horizon. test_df already carries ds + regressor columns.
future = test_df[["ds"] + REGRESSORS].copy()
forecast = m.predict(future)
yhat = forecast["yhat"].values
y_true = test_df["y"].values

prophet_report = regression_report(y_true, yhat, name="prophet-test")
prophet_report


In [ ]:
# Component decomposition: trend + seasonalities + regressor effects.
fig = m.plot_components(forecast)
plt.show()


## Local execution blocker

Authored but not executed end-to-end on the local `.venv` (Python 3.9). `pip install prophet` does not produce an importable module on this interpreter (likely a `cmdstanpy` / build-toolchain issue under py3.9). Prophet on 60k+ hourly rows also takes 10–30 min to fit on CPU; the cell already includes a 3-year-window fallback you can uncomment. **Run this notebook on the Jupyter server** with a working Prophet install.

## Hybrid: Prophet forecast as a feature for XGBoost

Append-only section. The existing Prophet results above stay untouched.
Feed Prophet's `yhat` as a single new column into a feature-augmented XGBoost that
also sees the full 87-feature matrix.

```
Raw price + regressors ──► Prophet ──► prophet_pred
                                          │
87 engineered features ──────────────────┐│
                                         ▼▼
                          XGBoost([87 features + prophet_pred] → price)
```

**Mild caveat:** the train+val Prophet predictions are *in-sample* (the same Prophet `m`
above was fit on train+val). That means XGBoost trains on Prophet outputs that have
implicit knowledge of the train+val global trend/seasonality. This is not a hard leak
(no specific future *value* informs a past prediction), but if hybrid test MAE looks
suspiciously better than train MAE, switch to a rolling-origin Prophet (fit on first
N-1 years, predict year N, advance) — slower but strictly honest.

In [ ]:
# Hybrid cell A — Prophet predictions on train+val (in-sample) and test (already computed)
import xgboost as xgb

forecast_tv = m.predict(train_val_df[["ds"] + REGRESSORS])
prophet_tv_yhat   = pd.Series(forecast_tv["yhat"].values, index=pd.to_datetime(train_val_df["ds"].values))
prophet_test_yhat = pd.Series(forecast["yhat"].values,   index=pd.to_datetime(test_df["ds"].values))
prophet_full = pd.concat([prophet_tv_yhat, prophet_test_yhat]).sort_index()
prophet_full.name = "prophet_pred"

# mat's index is tz-aware (UTC); prophet predictions are tz-naive. Re-localize for alignment.
prophet_full.index = prophet_full.index.tz_localize("UTC")
print(f"Prophet predictions: {len(prophet_full):,}  range: {prophet_full.index.min()} -> {prophet_full.index.max()}")

In [ ]:
# Hybrid cell B — build augmented feature matrix and train XGBoost on price
augmented = mat.loc[prophet_full.index].copy()
augmented["prophet_pred"] = prophet_full.values
features_aug = features + ["prophet_pred"]

aug_train = augmented[augmented.index < "2023-01-01"]
aug_val   = augmented[(augmented.index >= "2023-01-01") & (augmented.index < "2024-01-01")]
aug_test  = augmented[augmented.index >= "2024-01-01"]

X_train_h = aug_train[features_aug].ffill().fillna(0); y_train_h = aug_train[TARGET_REG]
X_val_h   = aug_val[features_aug].ffill().fillna(0);   y_val_h   = aug_val[TARGET_REG]
X_test_h  = aug_test[features_aug].ffill().fillna(0);  y_test_h  = aug_test[TARGET_REG]

print(f"{len(features_aug)} augmented features  train={len(X_train_h):,}  val={len(X_val_h):,}  test={len(X_test_h):,}")

xgb_aug = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective="reg:squarederror", tree_method="hist",
    early_stopping_rounds=30, eval_metric="rmse",
    n_jobs=-1, random_state=42,
)
xgb_aug.fit(X_train_h, y_train_h, eval_set=[(X_val_h, y_val_h)], verbose=10)

In [ ]:
# Hybrid cell C — report on test and rank of prophet_pred in feature importance
y_pred_test_h = xgb_aug.predict(X_test_h)
r_hybrid = regression_report(y_test_h.values, y_pred_test_h, "prophet-xgb-hybrid-test")

imp = pd.Series(xgb_aug.feature_importances_, index=features_aug).sort_values(ascending=False)
print("\nTop 20 features by importance:")
for name, val in imp.head(20).items():
    print(f"  {name:40s} {val:.4f}")
prophet_rank = list(imp.index).index("prophet_pred") + 1
print(f"\nprophet_pred rank: {prophet_rank} / {len(features_aug)}")

In [ ]:
from sklearn.metrics import average_precision_score
import numpy as np

# Variant 1: Prophet standalone
_y_true_dollar = np.asarray(y_true).reshape(-1)
_y_pred_dollar = np.asarray(yhat).reshape(-1)
_spike_pr_auc = average_precision_score(
    (_y_true_dollar > 200).astype(int),
    _y_pred_dollar,
)
print(f"\n=== Bake-off PR-AUC (regressor-as-score, threshold $200) ===")
print(f"  Spike PR-AUC [prophet-standalone]: {_spike_pr_auc:.3f}")

# Variant 2: Prophet + XGBoost hybrid
_y_true_h_dollar = np.asarray(y_test_h).reshape(-1)
_y_pred_h_dollar = np.asarray(y_pred_test_h).reshape(-1)
_spike_pr_auc_h = average_precision_score(
    (_y_true_h_dollar > 200).astype(int),
    _y_pred_h_dollar,
)
print(f"  Spike PR-AUC [prophet-xgb-hybrid]: {_spike_pr_auc_h:.3f}")

## Prophet — all 87 features variant
Re-runs Prophet under identical conditions (chronological split,
daily/weekly/yearly seasonality), this time passing every column from
`select_enhanced_features(mat)` as a regressor via `add_regressor`.
Expected to be slower (~30-60 min on CPU) and not necessarily better
than the curated 10-regressor variant — Prophet’s regressors are
additive linear terms, so many redundant or correlated features dilute
rather than improve the fit. Included for completeness in the bake-off.

In [ ]:
import re

def _safe(name: str) -> str:
    # Prophet's Stan formula chokes on spaces/parens. Sanitize → underscores.
    return re.sub(r"[^0-9A-Za-z_]+", "_", name).strip("_")

all_regressors = list(features)
rename_map = {c: _safe(c) for c in all_regressors}
safe_names = list(rename_map.values())
# detect collisions after sanitization (rare, but log if it happens)
if len(set(safe_names)) != len(safe_names):
    from collections import Counter
    dupes = [n for n, k in Counter(safe_names).items() if k > 1]
    raise ValueError(f"sanitized name collisions: {dupes}")

needed_cols_all = [TARGET_REG] + all_regressors
df_prophet_all = mat[needed_cols_all].dropna().copy()
df_prophet_all = df_prophet_all.rename(columns=rename_map).reset_index().rename(
    columns={df_prophet_all.index.name or "datetime": "ds", TARGET_REG: "y"})
df_prophet_all["ds"] = pd.to_datetime(df_prophet_all["ds"]).dt.tz_localize(None)

train_val_df_all = df_prophet_all[df_prophet_all["ds"] < test_start].copy()
test_df_all = df_prophet_all[df_prophet_all["ds"] >= test_start].copy()
print(f"all-features rows — train+val: {len(train_val_df_all):,}  test: {len(test_df_all):,}")
print(f"regressors used: {len(safe_names)}")

In [ ]:
from prophet import Prophet
print("Fitting Prophet on all 87 regressors; expect ~30-60 min on CPU...")
m_all = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=True,
    uncertainty_samples=0,   # skip posterior sampling for speed; we only need yhat
)
for r in safe_names:
    m_all.add_regressor(r)

m_all.fit(train_val_df_all)
print("Prophet (all 87) fit complete.")

future_all = test_df_all[["ds"] + safe_names].copy()
forecast_all = m_all.predict(future_all)
yhat_all = forecast_all["yhat"].values
y_true_all = test_df_all["y"].values

prophet_all_report = regression_report(
    y_true_all, yhat_all, name="prophet-all-87-test",
)

In [ ]:
from sklearn.metrics import average_precision_score
import numpy as np
_y_true_dollar = np.asarray(y_true_all).reshape(-1)
_y_pred_dollar = np.asarray(yhat_all).reshape(-1)
_spike_pr_auc_all = average_precision_score(
    (_y_true_dollar > 200).astype(int), _y_pred_dollar
)
print(f"\n=== Bake-off PR-AUC (prophet-all-87, regressor-as-score, $200) ===")
print(f"  Spike PR-AUC: {_spike_pr_auc_all:.3f}")

# Side-by-side
print("\n=== Prophet curated-10 vs all-87 summary ===")
print(f"  Curated 10 regressors: MAE ${prophet_report['mae']:.2f}  "
      f"RMSE ${prophet_report['rmse']:.2f}  "
      f"recall {prophet_report['spike_recall']:.1%}  "
      f"prec {prophet_report['spike_precision']:.1%}")
print(f"  All 87 regressors:     MAE ${prophet_all_report['mae']:.2f}  "
      f"RMSE ${prophet_all_report['rmse']:.2f}  "
      f"recall {prophet_all_report['spike_recall']:.1%}  "
      f"prec {prophet_all_report['spike_precision']:.1%}")